# 01 — Exploratory data analysis: five Olist datasets

This notebook explores the processed Parquet datasets without modifying them. The rules and severity classifications are documented in [Data Quality Rules](../docs/data-quality-rules.md). Run all cells in VS Code after creating the five processed Parquet files.


In [1]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

FILES = {
    "orders": "orders.parquet",
    "order_items": "order_items.parquet",
    "products": "products.parquet",
    "customers": "customers.parquet",
    "category_translation": "product_category_name_translation.parquet",
}

def find_root():
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "data" / "processed").is_dir():
            return path
    raise FileNotFoundError("Cannot find the project's data/processed directory")

ROOT = find_root()
missing = [filename for filename in FILES.values() if not (ROOT / "data" / "processed" / filename).is_file()]
if missing:
    raise FileNotFoundError(f"Missing Parquet files: {missing}")
con = duckdb.connect(database=":memory:")
for table, filename in FILES.items():
    path = str(ROOT / "data" / "processed" / filename).replace("'", "''")
    con.execute(f"CREATE VIEW {table} AS SELECT * FROM read_parquet('{path}')")
print("Five datasets loaded from the project directory")


Five datasets loaded from the project directory


## 1. Row counts and schema

In [2]:
overview = []
for name in FILES:
    count = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    schema = con.execute(f"DESCRIBE SELECT * FROM {name}").df()
    overview.append({"dataset": name, "rows": count, "columns": len(schema)})
    print(f"\n{name.upper()} — schema")
    display(schema)
display(pd.DataFrame(overview))



ORDERS — schema


,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None



ORDER_ITEMS — schema


,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,order_item_id,INTEGER,YES,None,None,None
2,product_id,VARCHAR,YES,None,None,None
3,seller_id,VARCHAR,YES,None,None,None
4,shipping_limit_date,TIMESTAMP,YES,None,None,None
5,price,DOUBLE,YES,None,None,None
6,freight_value,DOUBLE,YES,None,None,None



PRODUCTS — schema


,column_name,column_type,null,key,default,extra
0,product_id,VARCHAR,YES,None,None,None
1,product_category_name,VARCHAR,YES,None,None,None
2,product_name_lenght,INTEGER,YES,None,None,None
3,product_description_lenght,INTEGER,YES,None,None,None
4,product_photos_qty,INTEGER,YES,None,None,None
5,product_weight_g,DOUBLE,YES,None,None,None
6,product_length_cm,DOUBLE,YES,None,None,None
7,product_height_cm,DOUBLE,YES,None,None,None
8,product_width_cm,DOUBLE,YES,None,None,None



CUSTOMERS — schema


,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,YES,None,None,None
1,customer_unique_id,VARCHAR,YES,None,None,None
2,customer_zip_code_prefix,VARCHAR,YES,None,None,None
3,customer_city,VARCHAR,YES,None,None,None
4,customer_state,VARCHAR,YES,None,None,None



CATEGORY_TRANSLATION — schema


,column_name,column_type,null,key,default,extra
0,product_category_name,VARCHAR,YES,None,None,None
1,product_category_name_english,VARCHAR,YES,None,None,None


,dataset,rows,columns
0,orders,99441,8
1,order_items,112650,7
2,products,32951,9
3,customers,99441,5
4,category_translation,71,2


## 2. Missing values across all columns

In [3]:
null_rows = []
for name in FILES:
    cols = con.execute(f"DESCRIBE SELECT * FROM {name}").df()["column_name"].tolist()
    for col in cols:
        safe = col.replace('"', '""')
        null_count = con.execute(f'SELECT COUNT(*) FROM {name} WHERE "{safe}" IS NULL').fetchone()[0]
        null_rows.append({"dataset": name, "column": col, "null_count": null_count})
null_profile = pd.DataFrame(null_rows)
display(null_profile.sort_values(["dataset", "null_count"], ascending=[True, False]))


,dataset,column,null_count
29,category_translation,product_category_name,0
30,category_translation,product_category_name_english,0
24,customers,customer_id,0
25,customers,customer_unique_id,0
26,customers,customer_zip_code_prefix,0
27,customers,customer_city,0
28,customers,customer_state,0
8,order_items,order_id,0
9,order_items,order_item_id,0
10,order_items,product_id,0


## 3. Key uniqueness and referential integrity

In [4]:
queries = {
    "Duplicate order_id groups": "SELECT COUNT(*) FROM (SELECT order_id FROM orders GROUP BY 1 HAVING COUNT(*) > 1)",
    "Duplicate order-item key groups": "SELECT COUNT(*) FROM (SELECT order_id, order_item_id FROM order_items GROUP BY 1, 2 HAVING COUNT(*) > 1)",
    "Duplicate product_id groups": "SELECT COUNT(*) FROM (SELECT product_id FROM products GROUP BY 1 HAVING COUNT(*) > 1)",
    "Duplicate customer_id groups": "SELECT COUNT(*) FROM (SELECT customer_id FROM customers GROUP BY 1 HAVING COUNT(*) > 1)",
    "Duplicate category groups": "SELECT COUNT(*) FROM (SELECT product_category_name FROM category_translation GROUP BY 1 HAVING COUNT(*) > 1)",
    "Orders without matching customer": "SELECT COUNT(*) FROM orders o LEFT JOIN customers c ON o.customer_id=c.customer_id WHERE o.customer_id IS NOT NULL AND c.customer_id IS NULL",
    "Items without matching order": "SELECT COUNT(*) FROM order_items i LEFT JOIN orders o ON i.order_id=o.order_id WHERE i.order_id IS NOT NULL AND o.order_id IS NULL",
    "Items without matching product": "SELECT COUNT(*) FROM order_items i LEFT JOIN products p ON i.product_id=p.product_id WHERE i.product_id IS NULL OR p.product_id IS NULL",
}
display(pd.DataFrame([{"check": name, "count": con.execute(sql).fetchone()[0]} for name, sql in queries.items()]))


,check,count
0,Duplicate order_id groups,0
1,Duplicate order-item key groups,0
2,Duplicate product_id groups,0
3,Duplicate customer_id groups,0
4,Duplicate category groups,0
5,Orders without matching customer,0
6,Items without matching order,0
7,Items without matching product,0


## 4. Order status and chronology

In [5]:
display(con.execute("SELECT order_status, COUNT(*) AS orders FROM orders GROUP BY 1 ORDER BY 2 DESC").df())
display(con.execute("""
SELECT COUNT(*) FILTER (WHERE order_approved_at < order_purchase_timestamp) AS approval_before_purchase,
       COUNT(*) FILTER (WHERE order_delivered_carrier_date < order_approved_at) AS carrier_before_approval,
       COUNT(*) FILTER (WHERE order_delivered_customer_date < order_delivered_carrier_date) AS customer_before_carrier,
       COUNT(*) FILTER (WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL) AS delivered_missing_date
FROM orders
""").df())


,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


,approval_before_purchase,carrier_before_approval,customer_before_carrier,delivered_missing_date
0,0,1359,23,8


## 5. Product category translation coverage

In [6]:
display(con.execute("""
SELECT COUNT(DISTINCT p.product_category_name) AS categories_in_products,
       COUNT(DISTINCT p.product_category_name) FILTER (WHERE t.product_category_name IS NULL) AS categories_without_translation
FROM products p LEFT JOIN category_translation t ON p.product_category_name=t.product_category_name
""").df())
display(con.execute("SELECT product_category_name, COUNT(*) AS products FROM products GROUP BY 1 ORDER BY 2 DESC LIMIT 15").df())


,categories_in_products,categories_without_translation
0,73,2


,product_category_name,products
0,cama_mesa_banho,3029
1,esporte_lazer,2867
2,moveis_decoracao,2657
3,beleza_saude,2444
4,utilidades_domesticas,2335
5,automotivo,1900
6,informatica_acessorios,1639
7,brinquedos,1411
8,relogios_presentes,1329
9,telefonia,1134


## 6. Customers by state and city

In [7]:
display(con.execute("SELECT customer_state, COUNT(*) AS customers FROM customers GROUP BY 1 ORDER BY 2 DESC").df())
display(con.execute("SELECT customer_city, COUNT(*) AS customers FROM customers GROUP BY 1 ORDER BY 2 DESC LIMIT 15").df())


,customer_state,customers
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


,customer_city,customers
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521
5,campinas,1444
6,porto alegre,1379
7,salvador,1245
8,guarulhos,1189
9,sao bernardo do campo,938


## 7. Order item prices and freight

In [8]:
display(con.execute("""
SELECT COUNT(*) AS item_rows, COUNT(DISTINCT order_id) AS distinct_orders,
       MIN(price) AS min_price, ROUND(AVG(price), 2) AS avg_price, MAX(price) AS max_price,
       MIN(freight_value) AS min_freight, ROUND(AVG(freight_value), 2) AS avg_freight, MAX(freight_value) AS max_freight
FROM order_items
""").df())


,item_rows,distinct_orders,min_price,avg_price,max_price,min_freight,avg_freight,max_freight
0,112650,98666,0.85,120.65,6735.0,0.0,19.99,409.68


## 8. Interpretation

Review anomalies in context. Do not automatically remove source rows or alter timestamps. For all 31 executable rules, run `python src/validate_data.py` and `python src/generate_quality_report.py` from the project root. Counts above are computed dynamically and may change as the data pipeline evolves.

In [9]:
print("Exploration complete. No Parquet files were modified.")
con.close()


Exploration complete. No Parquet files were modified.
